In [ ]:
import xarray as xr
import pandas as pd
from dask.diagnostics import ProgressBar
from pathlib import Path
import math

# -----------------------------
# CONFIG
# -----------------------------
%run Data_Config.ipynb
print(f"NetCDF input directory: {input_dir}")
print(f"Zarr output directory: {zarr_output_dir}")
print(f"Date range: {DATE_RANGE}")

station_batch_size = 100   # Number of stations processed at once
target_time = pd.date_range("1970-01-01", "2023-12-31", freq="h")  # example range
chunks = {"station": station_batch_size, "time": -1}  # tweak as needed

# -----------------------------
# PREPROCESS FUNCTION
# -----------------------------
def preprocess_one_file(path):
    """Open a single NetCDF file, reindex time, assign station coordinate."""
    ds = xr.open_dataset(path, chunks={"time": -1})
    if 'input_station_id' in ds:
        ds = ds.drop_vars('input_station_id')
    
    # Reindex time to common target
    ds = ds.reindex(time=target_time)
    
    # Assign station ID (adjust if your attribute key differs)
    station_id = ds.attrs.get("station_id", Path(path).stem)
    ds = ds.assign_coords(station=("station", [station_id]))
    
    return ds



# -----------------------------
# FIX CHUNK SIZES
# -----------------------------
def ensure_uniform_chunks(ds, station_chunk=100, time_chunk=10000):
    """
    Force uniform chunks for station and time, store all other dims as a single chunk.
    """
    chunk_map = {}
    for dim, size in ds.dims.items():
        if dim == "station":
            chunk_map[dim] = station_chunk
        elif dim == "time":
            chunk_map[dim] = time_chunk
        else:
            # Small or flag-like dims → store whole
            chunk_map[dim] = size
    return ds.chunk(chunk_map)


# -----------------------------
# MAIN BATCHING LOOP
# -----------------------------
files = sorted(input_dir.glob("*.nc"))
num_batches = 1 #math.ceil(len(files) / station_batch_size)

for i in range(num_batches):
    batch_files = files[i*station_batch_size : (i+1)*station_batch_size]
    print(f"Processing batch {i+1}/{num_batches} ({len(batch_files)} files)")
    
    batch_datasets = []
    for f in batch_files:
        batch_datasets.append(preprocess_one_file(f))
    
    # Concatenate stations in this batch
    batch_ds = xr.concat(batch_datasets, dim="station")
    
    # Ensure uniform chunks before writing
    batch_ds = ensure_uniform_chunks(batch_ds)

batch_ds


In [ ]:
for var in batch_ds.data_vars:
    print(f"{var}: {batch_ds[var].chunks}")

In [ ]:
    # Write to Zarr
with ProgressBar():
    if i == 0:
        batch_ds.to_zarr(zarr_output_dir, mode="w")
    else:
        batch_ds.to_zarr(zarr_output_dir, mode="a", append_dim="station")

# Free memory
del batch_ds, batch_datasets

print("Zarr store written successfully!")


In [ ]:
# Open zarr store using xarray
ds_combined = xr.open_zarr(zarr_output_dir)
ds_combined

In [2]:
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

In [9]:
hadisd = HadISDIndex()
all_stations = hadisd.get_all_station_ids()
all_stations_ordered = sorted(all_stations)
print(f"Total number of stations: {len(all_stations_ordered)}")

hadisd.station = ["010010-99999"]
hadisd.station = all_stations_ordered[:10]

Total number of stations: 851


In [10]:
paths = hadisd.filesystem()
paths

{'010010-99999': PosixPath('/Users/joelmiller/HadISD_data/zarr/hadisd.3.4.0.2023f_19310101-20240101_010010-99999.zarr'),
 '010014-99999': PosixPath('/Users/joelmiller/HadISD_data/zarr/hadisd.3.4.0.2023f_19310101-20240101_010014-99999.zarr'),
 '010030-99999': PosixPath('/Users/joelmiller/HadISD_data/zarr/hadisd.3.4.0.2023f_19310101-20240101_010030-99999.zarr'),
 '010070-99999': PosixPath('/Users/joelmiller/HadISD_data/zarr/hadisd.3.4.0.2023f_19310101-20240101_010070-99999.zarr'),
 '010080-99999': PosixPath('/Users/joelmiller/HadISD_data/zarr/hadisd.3.4.0.2023f_19310101-20240101_010080-99999.zarr'),
 '010100-99999': PosixPath('/Users/joelmiller/HadISD_data/zarr/hadisd.3.4.0.2023f_19310101-20240101_010100-99999.zarr'),
 '010150-99999': PosixPath('/Users/joelmiller/HadISD_data/zarr/hadisd.3.4.0.2023f_19310101-20240101_010150-99999.zarr'),
 '010230-99999': PosixPath('/Users/joelmiller/HadISD_data/zarr/hadisd.3.4.0.2023f_19310101-20240101_010230-99999.zarr'),
 '010250-99999': PosixPath('/Use